In [ ]:
# 1. Import Required Libraries
!pip install -q flask scikit-learn pandas numpy requests

import json
import os
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import requests
from flask import Flask, jsonify, request
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

print("Libraries imported successfully.")

# 2. Load and Prepare Dataset
rng = np.random.default_rng(42)
feature_names = ["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]

profiles = {
    "class_0": np.array([1, 0, 1, 0, 0], dtype=int),
    "class_1": np.array([0, 1, 0, 1, 1], dtype=int),
    "class_2": np.array([1, 1, 1, 0, 1], dtype=int),
}

rows = []
for label, base in profiles.items():
    for _ in range(80):
        noise = rng.integers(0, 2, size=len(feature_names))
        features = np.clip(base + noise, 0, 2).astype(int)
        row = {name: int(value) for name, value in zip(feature_names, features)}
        row["label"] = label
        rows.append(row)

raw_df = pd.DataFrame(rows)
print(raw_df.head())
print(f"Dataset shape: {raw_df.shape}")
print(raw_df["label"].value_counts())

# 3. Data Preprocessing and Normalization

def preprocess_features(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()
    df_clean = df_clean.dropna()
    df_clean[feature_names] = df_clean[feature_names].astype(float)
    scaler = MinMaxScaler()
    scaled_values = scaler.fit_transform(df_clean[feature_names])
    df_clean[feature_names] = scaled_values
    return df_clean

processed_df = preprocess_features(raw_df)
print(processed_df.head())
print(processed_df.isnull().sum())

# 4. Train Naive Bayes Model
X = processed_df[feature_names]
y = processed_df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Model accuracy: {accuracy:.4f}")
print(classification_report(y_test, predictions, zero_division=0))

model_artifacts = {
    "feature_names": feature_names,
    "classes": list(model.classes_),
    "feature_count": len(feature_names),
    "model_type": "MultinomialNB",
    "accuracy": float(accuracy),
}

with open("naive_bayes_model.json", "w", encoding="utf-8") as f:
    json.dump(model_artifacts, f, ensure_ascii=False, indent=2)

print("Model saved to naive_bayes_model.json")

# 5. Create API Service Endpoint
app = Flask(__name__)


def normalize_features(raw_features: Any) -> List[float]:
    if isinstance(raw_features, str):
        raw_features = raw_features.strip().split(",")

    if not isinstance(raw_features, (list, tuple, np.ndarray)):
        raise ValueError("features must be a list/tuple/array or comma-delimited string")

    values = []
    for value in raw_features:
        try:
            values.append(float(value))
        except (TypeError, ValueError):
            raise ValueError(f"Invalid feature value: {value}")

    if len(values) != len(feature_names):
        raise ValueError(f"Expected {len(feature_names)} features, got {len(values)}")

    return values


def predict_from_input(raw_features: Any) -> Dict[str, Any]:
    values = normalize_features(raw_features)
    feature_vector = np.asarray([values], dtype=float)
    probabilities = model.predict_proba(feature_vector)[0]
    best_index = int(np.argmax(probabilities))
    predicted_label = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    return {
        "prediction": predicted_label,
        "probability": confidence,
        "probabilities": {
            label: float(probabilities[i])
            for i, label in enumerate(model.classes_)
        },
    }


@app.route("/", methods=["GET"])
def route_home():
    return jsonify(
        {
            "success": True,
            "status": 200,
            "message": "Naive Bayes Local AI API",
            "data": {
                "model": "naive_bayes",
                "endpoint": "/api/v1/predict",
                "health_status": "healthy",
            },
        }
    )


@app.route("/health", methods=["GET"])
def route_health():
    return jsonify(
        {
            "success": True,
            "status": 200,
            "message": "System ready",
            "data": {
                "model": "naive_bayes",
                "health_status": "healthy",
                "feature_count": len(feature_names),
            },
        }
    )


@app.route("/api/v1/predict", methods=["POST"])
def route_predict():
    payload = request.get_json(silent=True)
    if payload is None:
        return jsonify({
            "success": False,
            "status": 400,
            "message": "Request body must be JSON",
            "data": {}
        }), 400

    if "features" not in payload:
        return jsonify({
            "success": False,
            "status": 400,
            "message": "Missing 'features' field",
            "data": {}
        }), 400

    try:
        result = predict_from_input(payload["features"])
    except ValueError as exc:
        return jsonify({"success": False, "status": 400, "message": str(exc), "data": {}}), 400

    return jsonify({
        "success": True,
        "status": 200,
        "message": "Dự đoán Naive Bayes thành công",
        "data": {
            "model": "naive_bayes",
            "endpoint": "/api/v1/predict",
            "prediction": result["prediction"],
            "probability": round(float(result["probability"]), 4),
            "health_status": "healthy"
        }
    })

print("Flask app and endpoints created successfully.")

# 6. Test Prediction Endpoint
sample_payload = {"features": [1, 0, 1, 0, 0]}
print("Sample payload for testing:", sample_payload)

with app.test_client() as client:
    health_response = client.get("/health")
    print("Health status:", health_response.status_code, health_response.get_json())

    predict_response = client.post("/api/v1/predict", json=sample_payload)
    print("Prediction response:", predict_response.status_code, predict_response.get_json())

# 7. Health Check Monitoring
print("Health endpoint available at: http://localhost:5000/health")

# 8. Docker Configuration Setup
# Write Dockerfile
with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write("""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 5000
CMD [\"python\", \"app.py\"]
""")

# Write docker-compose.yml
with open("docker-compose.yml", "w", encoding="utf-8") as f:
    f.write("""services:
  naive-bayes-api:
    build: .
    container_name: naive-bayes-api
    ports:
      - \"5000:5000\"
    environment:
      - PYTHONUNBUFFERED=1
""")

# Write requirements.txt
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("""flask==3.0.3
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.5.2
requests==2.32.3
""")

# Write app.py for Docker/local execution
with open("app.py", "w", encoding="utf-8") as f:
    f.write('''import argparse
import numpy as np
from flask import Flask, jsonify, request
from sklearn.naive_bayes import MultinomialNB

feature_names = ["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]
profiles = {
    "class_0": np.array([1, 0, 1, 0, 0], dtype=int),
    "class_1": np.array([0, 1, 0, 1, 1], dtype=int),
    "class_2": np.array([1, 1, 1, 0, 1], dtype=int),
}

rng = np.random.default_rng(42)
rows = []
for label, base in profiles.items():
    for _ in range(80):
        noise = rng.integers(0, 2, size=len(feature_names))
        features = np.clip(base + noise, 0, 2).astype(int)
        row = {name: int(value) for name, value in zip(feature_names, features)}
        row["label"] = label
        rows.append(row)

X = np.array([[row[name] for name in feature_names] for row in rows], dtype=float)
y = np.array([row["label"] for row in rows])
model = MultinomialNB(alpha=1.0)
model.fit(X, y)

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"success": True, "status": 200, "message": "healthy", "data": {"health_status": "healthy"}})

@app.route("/api/v1/predict", methods=["POST"])
def predict():
    payload = request.get_json(silent=True)
    if payload is None or "features" not in payload:
        return jsonify({"success": False, "status": 400, "message": "Missing features"}), 400
    feature_values = payload["features"]
    if len(feature_values) != len(feature_names):
        return jsonify({"success": False, "status": 400, "message": "Invalid feature count"}), 400
    prob = model.predict_proba(np.asarray([feature_values], dtype=float))[0]
    cls = model.classes_[int(np.argmax(prob))]
    return jsonify({
        "success": True,
        "status": 200,
        "message": "Dự đoán Naive Bayes thành công",
        "data": {"prediction": cls, "probability": round(float(np.max(prob)), 4), "health_status": "healthy"}
    })

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="0.0.0.0")
    parser.add_argument("--port", type=int, default=5000)
    args = parser.parse_args()
    app.run(host=args.host, port=args.port, debug=False, use_reloader=False)
''')

print("Docker files and app.py generated successfully.")

# Final usage notes for Google Colab
print("Run this inside Colab to launch the app:")
print("from threading import Thread")
print("app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)")
print("Test using requests.post('http://localhost:5000/api/v1/predict', json={'features':[1,0,1,0,0]})")
print("Docker command: docker compose up --build")
print("Notebook setup complete.")